<a href="https://colab.research.google.com/github/jacobdawson093-tech/Montgomery-County-and-Bias-Incidents-Analysis/blob/main/eda/notebooks/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
url = "https://api.census.gov/data/2024/acs/acs5"
response = requests.get(url)
data = response.json()
url2 = "https://data.montgomerycountymd.gov/api/v3/views/7bhj-887p/query.json?app_token=8kUbrmGzgoxqe4z7C91iV3wmC"
response2 = requests.get(url2)
data2 = response2.json()

In [5]:
print("--- Initial Data Inspection ---")
# Convert the raw data into a pandas DataFrame, skipping metadata columns
df_bias = pd.DataFrame(data2)
df_bias = df_bias.drop(columns=[':id', ':version', ':created_at', ':updated_at'], errors='ignore')

print("DataFrame Head:")
print(df_bias.head())
print("\nDataFrame Info:")
df_bias.info()
print("\nDataFrame Shape:", df_bias.shape)

--- Initial Data Inspection ---
DataFrame Head:
          id            incident_date district    bias_code  \
0  260013584  2026-03-31T00:00:00.000       6D  Anti-Jewish   
1  260013065  2026-03-27T00:00:00.000       5D   Anti-Black   
2  260013221  2026-03-27T00:00:00.000       4D  Anti-Jewish   
3  260012859  2026-03-25T00:00:00.000       2D  Anti-Jewish   
4  260012953  2026-03-25T00:00:00.000       5D   Anti-Black   

                                 bias        status     victim_type  \
0                           Vandalism  Closed-Admin  School/College   
1                           Vandalism  Closed-Admin  School/College   
2                    Assault (simple)          Open   Individual(s)   
3                           Vandalism          Open         Society   
4  Verbal Intimidation/Simple Assault          Open   Individual(s)   

  no_of_victims no_of_suspects suspects_36_45_years_old suspects_55_years_old  \
0           NaN            NaN                      NaN          

In [6]:
print("\n--- Summary Statistics for Numerical Variables ---")
# Identify numerical columns for summary statistics
numerical_cols = [
    'no_of_victims', 'no_of_suspects', 'suspects_less_than_18_years',
    'suspects_18_35_years_old', 'suspects_36_45_years_old',
    'suspects_46_55_years_old', 'suspects_55_years_old'
]

for col in numerical_cols:
    # Convert columns to numeric, coercing errors to NaN
    df_bias[col] = pd.to_numeric(df_bias[col], errors='coerce')

print(df_bias[numerical_cols].describe())


--- Summary Statistics for Numerical Variables ---
       no_of_victims  no_of_suspects  suspects_less_than_18_years  \
count    1188.000000     1185.000000                   552.000000   
mean        1.252525        1.286920                     1.315217   
std         0.732126        0.784374                     0.831713   
min         1.000000        1.000000                     1.000000   
25%         1.000000        1.000000                     1.000000   
50%         1.000000        1.000000                     1.000000   
75%         1.000000        1.000000                     1.000000   
max        11.000000       10.000000                     9.000000   

       suspects_18_35_years_old  suspects_36_45_years_old  \
count                131.000000                 86.000000   
mean                   1.099237                  1.034884   
std                    0.324748                  0.184561   
min                    1.000000                  1.000000   
25%                  

In [7]:
print("\n--- Frequency Distributions for Categorical Variables ---")
categorical_cols = [
    'district', 'bias_code', 'bias_code_2', 'bias', 'status', 'victim_type'
]

for col in categorical_cols:
    print(f"\nFrequency distribution for '{col}':")
    print(df_bias[col].value_counts(dropna=False))
    print(f"Percentage distribution for '{col}':")
    print(df_bias[col].value_counts(normalize=True, dropna=False) * 100)


--- Frequency Distributions for Categorical Variables ---

Frequency distribution for 'district':
district
2D      565
4D      383
1D      303
5D      268
3D      244
6D      177
RCPD    116
GCPD    101
TPPD     26
Name: count, dtype: int64
Percentage distribution for 'district':
district
2D      25.881814
4D      17.544663
1D      13.879982
5D      12.276683
3D      11.177279
6D       8.108108
RCPD     5.313788
GCPD     4.626661
TPPD     1.191022
Name: proportion, dtype: float64

Frequency distribution for 'bias_code':
bias_code
Anti-Jewish                   766
Anti-Black                    695
Anti-Homosexual               154
Anti-Asian                    105
Anti-Hispanic                  98
Anti-Multi-Racial              66
Anti-Islamic                   60
Anti-White                     46
Anti-Male Homosexual           44
Anti-Transgender               40
Anti-Other Ethnicity           22
Anti-Arab                      11
Anti-Mental Disability         10
Anti-Catholic        

In [12]:
print("\n--- Time-based Analysis of Incident Dates ---")
# Convert 'incident_date' to datetime objects
df_bias['incident_date'] = pd.to_datetime(df_bias['incident_date'], errors='coerce')

# Drop rows where incident_date could not be parsed
df_bias.dropna(subset=['incident_date'], inplace=True)

# Extract year and month for time-based analysis
df_bias['incident_year'] = df_bias['incident_date'].dt.year
df_bias['incident_month'] = df_bias['incident_date'].dt.month

print("Incidents per Year:")
print(df_bias['incident_year'].value_counts().sort_index())

print("\nIncidents per Month (across all years):")
print(df_bias['incident_month'].value_counts().sort_index())


--- Time-based Analysis of Incident Dates ---
Incidents per Year:
incident_year
2016     98
2017    122
2018     93
2019    114
2020    117
2021    144
2022    160
2023    466
2024    485
2025    332
2026     52
Name: count, dtype: int64

Incidents per Month (across all years):
incident_month
1     174
2     229
3     235
4     181
5     212
6     183
7     107
8     128
9     173
10    194
11    201
12    166
Name: count, dtype: int64


## Consolidated Statistical Findings

### 1. Initial Data Overview (First 5 Rows)

In [20]:
display(df_bias.head())

,id,incident_date,district,bias_code,bias,status,victim_type,no_of_victims,no_of_suspects,suspects_36_45_years_old,suspects_55_years_old,unknown,suspects_less_than_18_years,bias_code_2,suspects_18_35_years_old,suspects_46_55_years_old,incident_year,incident_month
0,260013584,2026-03-31,6D,Anti-Jewish,Vandalism,Closed-Admin,School/College,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,3
1,260013065,2026-03-27,5D,Anti-Black,Vandalism,Closed-Admin,School/College,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,3
2,260013221,2026-03-27,4D,Anti-Jewish,Assault (simple),Open,Individual(s),2.0,2.0,1.0,1.0,Known,NaN,NaN,NaN,NaN,2026,3
3,260012859,2026-03-25,2D,Anti-Jewish,Vandalism,Open,Society,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026,3
4,260012953,2026-03-25,5D,Anti-Black,Verbal Intimidation/Simple Assault,Open,Individual(s),2.0,2.0,NaN,NaN,Known,1.0,NaN,NaN,NaN,2026,3


### 2. Summary Statistics for Numerical Variables

In [21]:
numerical_cols = [
    'no_of_victims', 'no_of_suspects', 'suspects_less_than_18_years',
    'suspects_18_35_years_old', 'suspects_36_45_years_old',
    'suspects_46_55_years_old', 'suspects_55_years_old'
]
display(df_bias[numerical_cols].describe())

,no_of_victims,no_of_suspects,suspects_less_than_18_years,suspects_18_35_years_old,suspects_36_45_years_old,suspects_46_55_years_old,suspects_55_years_old
count,1188.000000,1185.000000,552.000000,131.000000,86.000000,66.000000,83.000000
mean,1.252525,1.286920,1.315217,1.099237,1.034884,1.030303,1.036145
std,0.732126,0.784374,0.831713,0.324748,0.184561,0.172733,0.187784
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,11.000000,10.000000,9.000000,3.000000,2.000000,2.000000,2.000000


### 3. Frequency and Percentage Distributions for Categorical Variables

In [22]:
categorical_cols = [
    'district', 'bias_code', 'bias_code_2', 'bias', 'status', 'victim_type'
]

for col in categorical_cols:
    print(f"\n--- '{col}' Distribution ---")
    # Combine value counts and normalized percentages into a single DataFrame for display
    freq_df = pd.DataFrame({
        'Count': df_bias[col].value_counts(dropna=False),
        'Percentage': df_bias[col].value_counts(normalize=True, dropna=False) * 100
    })
    display(freq_df.sort_values(by='Count', ascending=False))


--- 'district' Distribution ---


,Count,Percentage
district,,
2D,565,25.881814
4D,383,17.544663
1D,303,13.879982
5D,268,12.276683
3D,244,11.177279
6D,177,8.108108
RCPD,116,5.313788
GCPD,101,4.626661
TPPD,26,1.191022



--- 'bias_code' Distribution ---


,Count,Percentage
bias_code,,
Anti-Jewish,766,35.089327
Anti-Black,695,31.836922
Anti-Homosexual,154,7.054512
Anti-Asian,105,4.809895
Anti-Hispanic,98,4.489235
Anti-Multi-Racial,66,3.023362
Anti-Islamic,60,2.748511
Anti-White,46,2.107192
Anti-Male Homosexual,44,2.015575



--- 'bias_code_2' Distribution ---


...

,Count,Percentage
bias,,
Vandalism,807,36.967476
Verbal Intimidation/Simple Assault,625,28.630325
Written Intimidation/Simple Assault,269,12.322492
Other,173,7.924874
Assault (simple),129,5.909299
Social Media,44,2.015575
Flyer Left Behind,39,1.786532
Physical Intimidation/Simple Assault,33,1.511681
Assault (physical),25,1.145213



--- 'status' Distribution ---


,Count,Percentage
status,,
Open,1017,46.587265
Closed-Admin,673,30.829134
N/A,191,8.749427
Closed-Arrest,123,5.634448
Inactive,94,4.306001
Closed-Exception,43,1.969766
UNF,29,1.328447
RTOJ,8,0.366468
NaN,5,0.229043



--- 'victim_type' Distribution ---


,Count,Percentage
victim_type,,
Individual(s),1186,54.328905
School/College,553,25.332112
Society,164,7.512597
Religious Organization,129,5.909299
Business/Financial Institution,91,4.168575
Government,27,1.236830
Other,19,0.870362
NaN,14,0.641319


### 4. Time-based Analysis

#### Incidents per Year

In [23]:
incidents_per_year_df = df_bias['incident_year'].value_counts().sort_index().reset_index()
incidents_per_year_df.columns = ['Year', 'Incident Count']
display(incidents_per_year_df)

,Year,Incident Count
0,2016,98
1,2017,122
2,2018,93
3,2019,114
4,2020,117
5,2021,144
6,2022,160
7,2023,466
8,2024,485
9,2025,332


#### Incidents per Month (across all years)

In [26]:
incidents_per_month_df = df_bias['incident_month'].value_counts().sort_index().reset_index()
incidents_per_month_df.columns = ['Month', 'Incident Count']
display(incidents_per_month_df)

,Month,Incident Count
0,1,174
1,2,229
2,3,235
3,4,181
4,5,212
5,6,183
6,7,107
7,8,128
8,9,173
9,10,194


In [27]:
import os
output_dir = 'eda_statistical_findings'
os.makedirs(output_dir, exist_ok=True)

df_bias.head().to_csv(os.path.join(output_dir, 'initial_data_head.csv'), index=False)
df_bias[numerical_cols].describe().to_csv(os.path.join(output_dir, 'numerical_summary.csv'))

for col in categorical_cols:
    freq_df = pd.DataFrame({
        'Count': df_bias[col].value_counts(dropna=False),
        'Percentage': df_bias[col].value_counts(normalize=True, dropna=False) * 100
    })
    freq_df.sort_values(by='Count', ascending=False).to_csv(os.path.join(output_dir, f'{col}_distribution.csv'))

incidents_per_year_df.to_csv(os.path.join(output_dir, 'incidents_per_year.csv'), index=False)
incidents_per_month_df.to_csv(os.path.join(output_dir, 'incidents_per_month.csv'), index=False)

print(f"All statistical findings saved to the '{output_dir}' directory.")

All statistical findings saved to the 'eda_statistical_findings' directory.
